# Chạy Ollama trên Google Colab qua Cloudflare (KHÔNG CẦN TÀI KHOẢN)
Notebook này giúp bạn đưa phần nặng nhất của AI Chatbot lên GPU miễn phí của Google Colab mà không cần đăng ký.
**HƯỚNG DẪN:** Bạn chỉ cần bấm nút **PLAY (Chạy ô này)** ở ngay bên trái ô code dưới đây và đợi máy xử lý xong tất cả.

In [ ]:
import os
import time
import subprocess
import threading
import re

print("1. Đang cài đặt công cụ giải nén zstd...")
os.system("apt-get install -y zstd > /dev/null 2>&1")

print("2. Đang tải Ollama từ GitHub (Định dạng tar.zst mới nhất)...")
os.system("wget -q -O ollama-linux-amd64.tar.zst https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tar.zst")

# Kiểm tra file tải về
check = subprocess.run(["file", "ollama-linux-amd64.tar.zst"], capture_output=True, text=True)
print(f"   File type: {check.stdout.strip()}")
size = subprocess.run(["ls", "-lh", "ollama-linux-amd64.tar.zst"], capture_output=True, text=True)
print(f"   Size: {size.stdout.strip()}")

print("3. Đang giải nén...")
os.system("tar --zstd -xf ollama-linux-amd64.tar.zst")

# Tìm file binary ollama sau khi giải nén
ollama_path = None
for root, dirs, files in os.walk("."):
    for f in files:
        if f == "ollama" and "test" not in root:
            full = os.path.join(root, f)
            check = subprocess.run(["file", full], capture_output=True, text=True)
            if "ELF" in check.stdout:
                ollama_path = full
                break
    if ollama_path:
        break

if not ollama_path:
    # Thử tìm trong /usr/local/bin hoặc bin/
    for p in ["bin/ollama", "./bin/ollama", "/usr/local/bin/ollama"]:
        if os.path.exists(p):
            ollama_path = p
            break

if not ollama_path:
    print("❌ Không tìm thấy binary ollama sau khi giải nén!")
    print("Nội dung thư mục hiện tại:")
    os.system("find . -name 'ollama*' -type f")
    raise SystemExit("Extract failed")

os.system(f"chmod +x {ollama_path}")
print(f"✅ Tìm thấy Ollama tại: {ollama_path}")

print("4. Đang khởi chạy máy chủ Ollama dưới nền...")
os.system(f"{ollama_path} serve > ollama.log 2>&1 &")
time.sleep(5)

# Health check
health = subprocess.run(["curl", "-s", "http://127.0.0.1:11434/"], capture_output=True, text=True)
if "Ollama" in health.stdout:
    print("✅ Ollama server đang chạy tốt!")
else:
    print(f"⚠️ Chưa sẵn sàng, đợi thêm 10s...")
    time.sleep(10)
    health = subprocess.run(["curl", "-s", "http://127.0.0.1:11434/"], capture_output=True, text=True)
    if "Ollama" in health.stdout:
        print("✅ Ollama server đang chạy tốt!")
    else:
        print("❌ Ollama không khởi động được! Log:")
        os.system("cat ollama.log")
        raise SystemExit("Ollama failed to start")

print("5. Đang tải đường hầm Cloudflare...")
os.system("wget -q -c -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
os.system("chmod +x cloudflared-linux-amd64")

def run_cloudflared():
    os.system("./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:11434 > cloudflare.log 2>&1")

threading.Thread(target=run_cloudflared, daemon=True).start()

print("6. Đang tạo link Public (Chờ 10 giây)...")
time.sleep(10)

url = None
try:
    with open('cloudflare.log', 'r') as f:
        content = f.read()
        match = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', content)
        if match:
            url = match.group(1)
except Exception as e:
    pass

print("\n==================================================")
if url:
    print(f"🔥 LINK KẾT NỐI (Hãy copy): {url}")
    print(f"\n👉 Dán vào file .env ở máy tính của bạn: OLLAMA_HOST={url}")
else:
    print("Đang tạo link, bạn hãy đợi vài giây rồi mở file 'cloudflare.log' ở cột bên trái của Colab để tự copy link nhé.")
print("==================================================\n")

print("7. Bắt đầu tải mô hình AI (Mất khoảng 3-5 phút)...")
print("-> Đang kéo nomic-embed-text...")
ret1 = os.system(f"{ollama_path} pull nomic-embed-text")
print(f"   Kết quả: {'✅ Thành công' if ret1 == 0 else '❌ Thất bại'}")

print("-> Đang kéo qwen2.5:7b...")
ret2 = os.system(f"{ollama_path} pull qwen2.5:7b")
print(f"   Kết quả: {'✅ Thành công' if ret2 == 0 else '❌ Thất bại'}")

if ret1 == 0 and ret2 == 0:
    print("\n✅ HOÀN TẤT! Google Colab đã sẵn sàng nhận tin nhắn từ máy của bạn.")
else:
    print("\n⚠️ Có lỗi khi tải model. Xem log ở trên.")

print("\n⚠️ KHÔNG tắt Tab này. Ô code sẽ chạy liên tục để giữ kết nối.")
while True:
    time.sleep(60)